# 🤟 ISL-CSLRT — SEN-Inspired CSLR Model
Based on: **SEN (Self-Emphasizing Network, AAAI 2023)** + **multi-scale temporal conv** + **auxiliary CTC (deep supervision)**

```
ResNet18 (frozen) → 512D
  → Multi-Scale Temporal Conv (kernels 1,3,5,7) → 512D
  → Temporal Self-Attention (emphasize discriminative frames)
  → BiLSTM Layer 1 → Auxiliary CTC Loss
  → BiLSTM Layer 2 → Auxiliary CTC Loss  
  → BiLSTM Layer 3 → Auxiliary CTC Loss
  → BiLSTM Layer 4 → Main CTC Loss
  Total loss = main + 0.5*(aux1+aux2+aux3)
```

In [ ]:
import os, re, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models, transforms
from PIL import Image
from collections import Counter
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#e6edf3','axes.labelcolor':'#e6edf3','axes.titlecolor':'#58a6ff',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','grid.color':'#21262d','font.size':11})

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE        = r'D:\ISL2\ISL_CSLRT_Corpus'
FRAMES_PATH = os.path.join(BASE, 'Frames_Sentence_Level')
FEATS_DIR   = os.path.join(BASE, 'resnet18_features')   # reuse cached features
CKPT_PATH   = os.path.join(BASE, 'best_sen_isl.pth')
VOCAB_PATH  = os.path.join(BASE, 'gloss_vocab.json')
TRAIN_S, VAL_S, TEST_S = ['1','2','3','4','5'], ['6'], ['7']

torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE=='cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## Phase 1 — Vocabulary

In [ ]:
def tokenize(text):
    if not text or pd.isna(text): return []
    return [w for w in re.sub(r"[^a-z0-9 ']", ' ', str(text).lower().strip()).split() if w]

if os.path.exists(VOCAB_PATH):
    vocab = json.load(open(VOCAB_PATH))
else:
    all_toks = []
    for sent in os.listdir(FRAMES_PATH): all_toks.extend(tokenize(sent))
    vocab = {'<blank>':0}
    for i,t in enumerate(sorted(set(all_toks)),1): vocab[t]=i
    json.dump(vocab, open(VOCAB_PATH,'w'), indent=2)

idx_to_tok = {v:k for k,v in vocab.items()}
VOCAB_SIZE = len(vocab)
RAND_BASE  = math.log(VOCAB_SIZE)
print(f'Vocab size     : {VOCAB_SIZE}')
print(f'Random baseline: {RAND_BASE:.3f}  (target: beat this)')

---
## Phase 2 — Extract & Cache ResNet18 Features
> Skipped if `resnet18_features/` already exists from previous run.

In [ ]:
from tqdm import tqdm

resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
feat_ext = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE).eval()
for p in feat_ext.parameters(): p.requires_grad=False

preprocess = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

existing = sum(len([f for f in os.listdir(os.path.join(FEATS_DIR,sp)) if f.endswith('.npy')])
    for sp in ['train','val','test'] if os.path.isdir(os.path.join(FEATS_DIR,sp)))

folder_to_sent = {f:' '.join(tokenize(f)) for f in sorted(os.listdir(FRAMES_PATH))}

if existing > 100:
    print(f'✅ ResNet18 features cached ({existing} files). Skipping.')
else:
    for sp in ['train','val','test']: os.makedirs(os.path.join(FEATS_DIR,sp), exist_ok=True)
    done, skip = 0, 0
    for sent_folder in tqdm(sorted(os.listdir(FRAMES_PATH)), desc='Sentences'):
        sent_path = os.path.join(FRAMES_PATH, sent_folder)
        sent_txt  = folder_to_sent[sent_folder]
        for signer in sorted(os.listdir(sent_path)):
            signer_path = os.path.join(sent_path, signer)
            if not os.path.isdir(signer_path): continue
            split = 'train' if signer in TRAIN_S else ('val' if signer in VAL_S else 'test')
            safe  = re.sub(r'[^\w]','_',f'{sent_folder}_s{signer}')[:100]
            npy_o = os.path.join(FEATS_DIR,split,f'{safe}.npy')
            lbl_o = os.path.join(FEATS_DIR,split,f'{safe}_label.txt')
            if os.path.exists(npy_o): done+=1; continue
            imgs = sorted([f for f in os.listdir(signer_path) if f.lower().endswith(('.jpg','.png'))])
            if not imgs: skip+=1; continue
            feats, batch = [], []
            for img_f in imgs:
                try: batch.append(preprocess(Image.open(os.path.join(signer_path,img_f)).convert('RGB')))
                except: continue
                if len(batch)==32:
                    with torch.no_grad(): feats.append(feat_ext(torch.stack(batch).to(DEVICE)).squeeze(-1).squeeze(-1).cpu().numpy())
                    batch=[]
            if batch:
                with torch.no_grad(): feats.append(feat_ext(torch.stack(batch).to(DEVICE)).squeeze(-1).squeeze(-1).cpu().numpy())
            if not feats: skip+=1; continue
            np.save(npy_o, np.concatenate(feats,0)); open(lbl_o,'w').write(sent_txt); done+=1
    print(f'Done: {done} saved, {skip} skipped.')

for sp in ['train','val','test']:
    d=os.path.join(FEATS_DIR,sp)
    if os.path.isdir(d): print(f'  {sp}: {len([f for f in os.listdir(d) if f.endswith(".npy")])} files')

---
## Phase 3 — Dataset with Strong Augmentation

In [ ]:
def aug_noise(x):       return x + np.random.randn(*x.shape).astype(np.float32)*0.01
def aug_time_mask(x):
    x=x.copy()
    for _ in range(3):
        t0=random.randint(0,max(x.shape[0]-5,0)); x[t0:t0+5]=0
    return x
def aug_speed(x):
    T=x.shape[0]; r=random.uniform(0.8,1.2)
    idx=np.round(np.linspace(0,T-1,max(int(T*r),4))).astype(int).clip(0,T-1)
    return x[idx]
def aug_feat_dropout(x, p=0.1):
    mask=(np.random.rand(x.shape[1])>p).astype(np.float32)
    return x*mask
def augment(x, train=True):
    if not train: return x
    if random.random()<0.6: x=aug_noise(x)
    if random.random()<0.5: x=aug_time_mask(x)
    if random.random()<0.6: x=aug_speed(x)
    if random.random()<0.4: x=aug_feat_dropout(x)
    return np.nan_to_num(x)

class ISLDataset(Dataset):
    def __init__(self, split_dir, vocab, max_len=300, training=False):
        self.samples=[]; self.max_len=max_len; self.training=training
        for f in sorted(os.listdir(split_dir)):
            if not f.endswith('.npy'): continue
            lb=os.path.join(split_dir,f[:-4]+'_label.txt')
            if not os.path.exists(lb): continue
            txt=open(lb,encoding='utf-8').read().strip()
            ids=[vocab[t] for t in tokenize(txt) if t in vocab]
            if ids: self.samples.append((os.path.join(split_dir,f),ids,txt))
        print(f'  {split_dir.split(os.sep)[-1]:6s}: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)
    def __getitem__(self,idx):
        np_,ids,_=self.samples[idx]
        x=augment(np.nan_to_num(np.load(np_).astype(np.float32)),self.training)
        if x.shape[0]>self.max_len:
            i=np.round(np.linspace(0,x.shape[0]-1,self.max_len)).astype(int); x=x[i]
        return torch.tensor(x,dtype=torch.float32), torch.tensor(ids,dtype=torch.long)

def collate_fn(batch):
    xs,ys=zip(*batch)
    xl=torch.tensor([x.shape[0] for x in xs],dtype=torch.long)
    yl=torch.tensor([y.shape[0] for y in ys], dtype=torch.long)
    return pad_sequence(xs,batch_first=True),pad_sequence(ys,batch_first=True),xl,yl

print('Loading datasets...')
train_ds=ISLDataset(os.path.join(FEATS_DIR,'train'),vocab,training=True)
val_ds  =ISLDataset(os.path.join(FEATS_DIR,'val'),  vocab,training=False)
train_loader=DataLoader(train_ds,batch_size=8,shuffle=True, collate_fn=collate_fn,num_workers=0,pin_memory=(DEVICE=='cuda'))
val_loader  =DataLoader(val_ds,  batch_size=8,shuffle=False,collate_fn=collate_fn,num_workers=0,pin_memory=(DEVICE=='cuda'))
print(f'Feature dim: 512 | Vocab: {VOCAB_SIZE}')

---
## Phase 4 — SEN-Inspired Model
> **Research basis:**
> - *SEN (AAAI 2023)*: Temporal self-attention to emphasize discriminative frames
> - *Multi-scale temporal conv*: capture motion at different time scales (kernels 1,3,5,7)
> - *Auxiliary CTC*: deep supervision at each BiLSTM layer = better gradient flow
> - *4-layer BiLSTM*: deeper temporal modeling as requested

In [ ]:
# ── Multi-Scale Temporal Conv (MSTC) ──────────────────────────────
class MSTC(nn.Module):
    """Multi-Scale Temporal Convolution — processes frames at 4 temporal scales.
    Captures both fine-grained hand movements and coarse sentence-level motion."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        branch_ch = out_ch // 4
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch, branch_ch, k, padding=k//2),
                nn.BatchNorm1d(branch_ch), nn.ReLU()
            ) for k in [1, 3, 5, 7]
        ])
        self.proj = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, 1), nn.BatchNorm1d(out_ch), nn.ReLU()
        )
        # Residual if dims match
        self.res = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):              # [B, T, C]
        xt = x.permute(0,2,1)         # [B, C, T]
        out = torch.cat([b(xt) for b in self.branches], dim=1)  # [B, out_ch, T]
        out = self.proj(out) + self.res(xt)
        return out.permute(0,2,1)      # [B, T, out_ch]

# ── Temporal Self-Attention (SEN-style frame emphasis) ─────────────
class TemporalSelfAttn(nn.Module):
    """Learns weights for each frame: emphasizes sign-relevant frames,
    suppresses transition/background frames. From SEN (AAAI 2023)."""
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(dim, dim//4), nn.Tanh(),
            nn.Linear(dim//4, 1)
        )
    def forward(self, x):              # [B, T, C]
        w = torch.sigmoid(self.score(x))   # [B, T, 1]
        return x * w                   # emphasize informative frames

# ── Main Model ─────────────────────────────────────────────────────
class SEN_CSLR(nn.Module):
    """
    SEN-Inspired CSLR model with:
    - Multi-Scale Temporal Conv (MSTC)
    - Temporal Self-Attention (frame emphasis)
    - 4-layer BiLSTM (as requested)
    - Auxiliary CTC at each BiLSTM layer (deep supervision)
    """
    def __init__(self, feat_dim=512, hidden=256, vocab_size=50, p=0.4):
        super().__init__()

        # Input projection + MSTC
        self.input_proj = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(p)
        )
        self.mstc = MSTC(512, 512)
        self.temporal_attn = TemporalSelfAttn(512)
        self.drop = nn.Dropout(p)

        # 4 BiLSTM layers
        self.lstm_layers = nn.ModuleList([
            nn.LSTM(512 if i==0 else hidden*2, hidden, 1,
                    batch_first=True, bidirectional=True)
            for i in range(4)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden*2) for _ in range(4)])
        self.drops = nn.ModuleList([nn.Dropout(p) for _ in range(4)])

        # CTC head for each layer (deep supervision)
        self.ctc_heads = nn.ModuleList([
            nn.Linear(hidden*2, vocab_size) for _ in range(4)
        ])

        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'weight_hh' in name: nn.init.orthogonal_(p)
            elif 'weight_ih' in name: nn.init.xavier_uniform_(p)
            elif 'bias' in name: nn.init.zeros_(p)

    def forward(self, x):              # [B, T, 512]
        x = self.input_proj(x)         # [B, T, 512]
        x = self.mstc(x)               # multi-scale temporal features
        x = self.temporal_attn(x)      # emphasize discriminative frames
        x = self.drop(x)

        logits_all = []
        for lstm, norm, drop, head in zip(self.lstm_layers, self.norms, self.drops, self.ctc_heads):
            x, _ = lstm(x)
            x = drop(norm(x))
            logits_all.append(head(x))  # auxiliary CTC output per layer

        return logits_all   # list of 4 × [B, T, V]


model = SEN_CSLR(feat_dim=512, hidden=256, vocab_size=VOCAB_SIZE, p=0.4).to(DEVICE)
n = sum(p.numel() for p in model.parameters())
print(f'Model: SEN_CSLR  |  Params: {n:,}')
print(f'BiLSTM layers: 4  |  Hidden: 256 each  |  Multi-scale kernels: 1,3,5,7')
print(f'Auxiliary CTC heads: 4 (deep supervision)')

---
## Phase 5 — Training with Deep Supervision

In [ ]:
EPOCHS=150; LR=5e-4; PATIENCE=25
# Auxiliary loss weights (later layers get higher weight)
AUX_W = [0.1, 0.2, 0.3, 1.0]   # layers 1,2,3,4

ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True, reduction='mean')
optimizer= torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)

def lr_fn(ep):
    warmup=10
    if ep<warmup: return (ep+1)/warmup
    return 0.5*(1+math.cos(math.pi*(ep-warmup)/max(EPOCHS-warmup,1)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_fn)


def run_epoch(loader, train=True):
    model.train(train)
    total, n = 0.0, 0
    for X,Y,xl,yl in loader:
        X,Y=X.to(DEVICE),Y.to(DEVICE)
        if train: optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits_all = model(X)          # list of 4 × [B,T,V]
            loss = torch.tensor(0.0, device=DEVICE)
            for logits, w in zip(logits_all, AUX_W):
                lp = F.log_softmax(logits, dim=-1).permute(1,0,2)  # [T,B,V]
                il = xl.clamp(max=lp.shape[0])
                l  = ctc_loss(lp, Y, il, yl)
                if not (torch.isnan(l) or torch.isinf(l)):
                    loss = loss + w * l
        if train and not (torch.isnan(loss) or torch.isinf(loss)):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        if not torch.isnan(loss): total+=loss.item(); n+=1
    return total/max(n,1)


train_h, val_h = [], []
best_val, no_imp = float('inf'), 0

print(f'SEN_CSLR Training | {EPOCHS}ep | LR={LR} | patience={PATIENCE}')
print(f'Aux CTC weights: {AUX_W}  (deep supervision on all 4 BiLSTM layers)')
print(f'Random CTC baseline: {RAND_BASE:.3f}')
print('─'*75)

for ep in range(1,EPOCHS+1):
    tl=run_epoch(train_loader,True)
    vl=run_epoch(val_loader,  False)
    scheduler.step()
    train_h.append(tl); val_h.append(vl)
    lr_now=optimizer.param_groups[0]['lr']
    star=''
    if vl<best_val:
        best_val=vl; no_imp=0
        torch.save({'epoch':ep,'model':model.state_dict(),'vocab':vocab,'val_loss':vl}, CKPT_PATH)
        star='  ✅'
    else: no_imp+=1
    print(f'Ep {ep:3d}/{EPOCHS}  tr={tl:.4f}  vl={vl:.4f}  gap={vl-tl:+.3f}  lr={lr_now:.2e}{star}')
    if no_imp>=PATIENCE: print(f'Early stop ep {ep}'); break

print(f'\nBest val: {best_val:.4f}  |  Random: {RAND_BASE:.4f}')
print('✅ Learning!' if best_val<RAND_BASE else '⚠️  Stuck — check labels or data.')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,5))
fig.suptitle('SEN_CSLR — ResNet18+MSTC+TemporalAttn+4×BiLSTM+AuxCTC', color='#58a6ff',fontsize=13)
axes[0].plot(train_h,color='#3fb950',lw=2,label='Train')
axes[0].plot(val_h,color='#f85149',lw=2,label='Val')
axes[0].axhline(RAND_BASE,color='#ffa657',ls='--',lw=1.5,label=f'Random ({RAND_BASE:.2f})')
axes[0].set_title('CTC Loss'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
gap=[v-t for t,v in zip(train_h,val_h)]
axes[1].bar(range(len(gap)),gap,color=['#f85149' if g>0 else '#3fb950' for g in gap],width=1,alpha=0.7)
axes[1].axhline(0,color='white',lw=1)
axes[1].set_title('Val−Train Gap (red=overfit)'); axes[1].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE,'sen_training.png'),dpi=150,bbox_inches='tight')
plt.show()

---
## Phase 6 — Evaluation (WER)

In [ ]:
ckpt=torch.load(CKPT_PATH,map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print(f'Best model: epoch={ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}')

def greedy_decode(lp,blank=0):
    ids=lp.argmax(-1).cpu().numpy(); prev,out=-1,[]
    for i in ids:
        if i!=prev and i!=blank: out.append(idx_to_tok.get(int(i),'<unk>'))
        prev=i
    return out

def wer(ref,hyp):
    d=np.zeros((len(ref)+1,len(hyp)+1),int)
    for i in range(len(ref)+1): d[i][0]=i
    for j in range(len(hyp)+1): d[0][j]=j
    for i in range(1,len(ref)+1):
        for j in range(1,len(hyp)+1):
            d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+(0 if ref[i-1]==hyp[j-1] else 1))
    return d[len(ref)][len(hyp)]/max(len(ref),1)

model.eval(); wers,exact,examples=[],0,[]
with torch.no_grad():
    for np_,ids,label in val_ds.samples:
        x=np.nan_to_num(np.load(np_).astype(np.float32))
        logits_all=model(torch.tensor(x).unsqueeze(0).to(DEVICE))
        lp=F.log_softmax(logits_all[-1][0],dim=-1)   # use final layer
        pred=greedy_decode(lp); truth=tokenize(label)
        w=wer(truth,pred); wers.append(w)
        if pred==truth: exact+=1
        if len(examples)<10: examples.append((label,' '.join(pred) or '<empty>',f'{w*100:.0f}%'))

mwer=np.mean(wers)*100; acc=exact/max(len(val_ds),1)*100
print(f'\n{"═"*60}')
print(f'  WER          : {mwer:.2f}%')
print(f'  Sequence Acc : {acc:.2f}%')
print(f'{"═"*60}\n')
print(f'{"REFERENCE":40s}  {"PREDICTION":40s}  WER')
print('─'*90)
for r,h,w in examples: print(f'{r[:40]:40s}  {h[:40]:40s}  {w}')

---
## Architecture Summary

| Component | Design | Research basis |
|---|---|---|
| **Backbone** | ResNet18 (frozen, pre-extracted) | Transfer learning from ImageNet |
| **MSTC** | 1D Conv with kernels 1,3,5,7 → concat | Multi-scale temporal features |
| **Frame attention** | Temporal Self-Attention (sigmoid) | SEN (AAAI 2023) |
| **Temporal** | **4-layer BiLSTM** × 256 hidden | Deep temporal context |
| **Loss** | Main CTC (w=1.0) + 3 aux CTCs (w=0.1,0.2,0.3) | Deep supervision |
| **Augmentation** | Noise + time mask + speed + feat dropout | Compensates for small dataset |
| **Training** | AdamW + warmup(10ep) + cosine 150ep | Stable convergence |